In [2]:
from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolAlign
from rdkit.Chem import ChemicalFeatures
from rdkit.RDConfig import RDDataDir
import os
import pandas as pd
#reading all ligands for Micobacterium tuberculosis
ligands = pd.read_excel('Mtb.xlsx', sheet_name=1)
#Name SMILES IC50 of all aurachin D homologues
ligands_Qloop = ligands[ligands['Binding site'] == 'Q-Loop']
ligands_Qloop_needed_columns = ligands_Qloop[['Name', 'SMILES', 'IC50 μM']]
#taking 1/3 of the data as training set for a model
training_set = ligands_Qloop_needed_columns[ligands_Qloop_needed_columns['IC50 μM'] < 0.3]

print(training_set)


            Name                                             SMILES  IC50 μM
2     Aurachin D  CC1=C(C(=O)C2=CC=CC=C2N1)C/C=C(\C)/CC/C=C(\C)/...    0.150
3   CK-3-22 (1T)  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)nc2)[nH]c3cccc...    0.140
8        MTD-403     Cc4c(c2ccc(N1CCCCC1)cc2)[nH]c3cc(F)cc(F)c3c4=O    0.270
9        CK-2-88          Cc4c(c2ccc(Cc1ccccc1)cc2)[nH]c3ccccc3c4=O    0.020
11       CK-2-63  Cc4c(c2ccc(Oc1ccc(OC(F)(F)F)cc1)cc2)[nH]c3cccc...    0.003
12        PG-203  Cc2[nH]c1ccccc1c(=O)c2c4ccc(Oc3ccc(OC(F)(F)F)c...    0.070
15          LT-9        O=c3cc(c2ccc(Cc1ccc(F)cc1)cc2)[nH]c4ccccc34    0.100
16        GN-171  CCOC(=O)c4c(c2ccc(Cc1ccc(OC(F)(F)F)cc1)cc2)[nH...    0.250
18       SL-2-25  Cc4c(c2ccc(c1ccc(OC(F)(F)F)cc1)nc2)[nH]c3ccccc...    0.290
19     WDH-1U-10  CCOC(=O)c4c(c2ccc(c1ccc(Cl)cc1)cc2)[nH]c3ccccc...    0.012
22      WDH-2G-6  CC(C)c4c(c2cnn(Cc1ccc(OC(F)(F)F)cc1)c2)[nH]c3c...    0.082


In [ ]:
#this function is not needed anymore and functuonality is placed in Conformer generator.ipynb
def Selecting_All_Confirmations(SMILES):
    #Creating molecule and preparing for conformer generation
    mol = Chem.MolFromSmiles(SMILES)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol)
    params = AllChem.ETKDGv3()
    print("mol created")
    #parameters of conforment simulation
    params.numThreads = 0
    params.randomSeed = 42
    params.pruneRmsThresh = -1 
    #2000 confirmantions
    confs = AllChem.EmbedMultipleConfs(
        mol,
        numConfs=1000,
        params=params
    )
    print('conformers generated')
    #conformer optimization
    results = AllChem.MMFFOptimizeMoleculeConfs(
        mol,
        mmffVariant='MMFF94s',
        maxIters=500
    )
    #energy calculations
    energies = [(i, res[1]) for i, res in enumerate(results)]
    energies.sort(key=lambda x: x[1])
    #energy window for conformers
    Emin = energies[0][1]
    energy_window = 5.0

    lowE = [idx for idx, E in energies if E <= Emin + energy_window]
    print('energy window applied')
    #rmsd filtration
    # rmsd filtration
    selected = []
    rmsd_cutoff = 0.7
    total = len(lowE)

    print(f"Starting RMSD filtering for {total} conformers...")

    for i, idx in enumerate(lowE):
        # Progress bar
        progress = (i + 1) / total * 100
        print(f"\rProgress: [{int(progress/2)*'=':50}] {progress:.1f}% ({i+1}/{total})", end="", flush=True)

        if all(
            rdMolAlign.GetBestRMS(
                mol, mol,
                prbId=idx,
                refId=s,
                symmetrizeConjugatedTerminalGroups=False
            ) > rmsd_cutoff
            for s in selected
        ):
            selected.append(idx)
    
    print("\nRMSD filter applied.")
#return mols with needed conformers
    new_mol = Chem.Mol(mol)
    new_mol.RemoveAllConformers()
    for idx in selected:
        new_mol.AddConformer(mol.GetConformer(idx), assignId=True)
    print('confirmations saved')
        
    return new_mol


In [ ]:
fdefName = os.path.join(RDDataDir, 'BaseFeatures.fdef')
featFactory = ChemicalFeatures.BuildFeatureFactory(fdefName)

training_set['conf_feats'] = training_set['molecule'].apply(lambda x: extract_pharmacophore_data(x, featFactory))

mol created
conformers generated
energy window applied
Starting RMSD filtering for 904 conformers...
Progress: [                                                  ] 1.9% (17/904)

In [ ]:
"""
Pharmacophore Pipeline:
  1. Load all conformers from Conformers/ folder (.sdf files)
  2. ED (Energy-based / Distance) alignment to a template molecule
  3. RMSD filtration to remove redundant conformers
  4. Build pharmacophore model using feature factory
  5. Save pharmacophore model to .xml file

Template molecule: CK_2_63.sdf
"""

import os
import sys
import glob
import numpy as np
from xml.etree.ElementTree import Element, SubElement, ElementTree, indent
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS
from rdkit.Chem.Pharm3D import Pharmacophore, EmbedLib
from rdkit.Chem import ChemicalFeatures
from rdkit.RDKit import RDConfig

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CONFORMERS_DIR   = "conformers"          # folder with .sdf conformer files
TEMPLATE_NAME    = "CK_2_63.sdf"        # template molecule filename
RMSD_THRESHOLD   = 1.0                  # Å — conformers closer than this are pruned
OUTPUT_XML       = "pharmacophore_model.xml"

# Feature factory definition (inline SMARTS-based factory)
FEATURE_FACTORY_FDEF = """
AtomType NDonor [N;!H0;v3,v4&+1]
AtomType NDonor [$([N;H2&+0][$([C,a]);!$([C,a](=O))])]
AtomType NDonor [$([N;H1&+0]([$([C,a]);!$([C,a](=O))])[$([C,a]);!$([C,a](=O))])]
AtomType NDonor [n;H1;+0]
AtomType NAcceptor [N;H0;$(N(-C(=O)))]
AtomType NAcceptor [$([N;H0]#[C&v4])]
AtomType NAcceptor [#7;H0;+0;D2:1]-[#6;+0]-[#7;H0;+0;D2:2]
AtomType ChalcDonor [O,S;H1;+0]
AtomType ChalcAcceptor [#8&!$([OH]);$([R0])]
AtomType ChalcAcceptor [#8;H0;+0;$([#8]~[#6]);!$([#8]~[#6]~[#8])]

Family Donor NDonor,ChalcDonor
Family Acceptor NAcceptor,ChalcAcceptor
Family Aromatic a1aaaaa1,a1aaaa1
Family Hydrophobe [c,s,S&H0&v2,$([D3&!Ring1]cc),$([D4&!Ring1]ccc),$([D3&!Ring1][#6]~[#6]),C&v4&!$([CH2]~[*;!#6])&!$([CH3]~[*;!#6])&!$([CH2]~[#8,#7,#16,#15,F,Cl])]
Family PosIonizable [+,$([N;H2&+0][$([C,a]);!$([C,a](=O))]),$([N;H1&+0]([C;!$(C(=O))])[C;!$(C(=O))]),$([N;H0&+0]([C;!$(C(=O))])([C;!$(C(=O))])[C;!$(C(=O))]),$([n;+0;H0]1cccc1),$([$([N;H0](=C)),$([N;H1](=C))]-[!#6]),$([#7;+0;H0]~[#6;+0;H0]~[#7;+0;H0]),$([N;H1,H2;+0;$([N]([#6])[#6])]C(=N)N)]
Family NegIonizable [C,S](=[O,S,P])-[O;H1,H0&-1]
"""

# ─────────────────────────────────────────────────────────────────────────────
# 1. FEATURE FACTORY  (uses your provided extraction function)
# ─────────────────────────────────────────────────────────────────────────────

def build_factory(fdef_string: str) -> ChemicalFeatures.MolChemicalFeatureFactory:
    """Build a MolChemicalFeatureFactory from an fdef string."""
    import tempfile
    with tempfile.NamedTemporaryFile(mode="w", suffix=".fdef", delete=False) as fh:
        fh.write(fdef_string)
        tmp_path = fh.name
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(fdef_string)
    os.unlink(tmp_path)
    return factory


def extract_pharmacophore_data(mol, factory):
    """Extract pharmacophore features for every conformer in mol."""
    all_conf_data = {}
    feats = factory.GetFeaturesForMol(mol)
    for conf in mol.GetConformers():
        conf_id = conf.GetId()
        conf_feats = []
        for f in feats:
            conf_feats.append({
                'family': f.GetFamily(),
                'type':   f.GetType(),
                'pos':    list(f.GetPos(confId=conf_id))
            })
        all_conf_data[conf_id] = conf_feats
    return all_conf_data


# ─────────────────────────────────────────────────────────────────────────────
# 2. LOAD MOLECULES
# ─────────────────────────────────────────────────────────────────────────────

def load_mol_from_sdf(path: str):
    """Load the first molecule from an SDF file; return None on failure."""
    suppl = Chem.SDMolSupplier(path, removeHs=False)
    for mol in suppl:
        if mol is not None:
            return mol
    return None


def load_all_conformers(folder: str, template_name: str):
    """
    Load every .sdf in folder into a list of (name, mol) tuples.
    Returns (template_mol, [(name, mol), ...]).
    """
    sdf_files = sorted(glob.glob(os.path.join(folder, "*.sdf")))
    if not sdf_files:
        raise FileNotFoundError(f"No .sdf files found in '{folder}'")

    template_mol = None
    molecules    = []

    for path in sdf_files:
        name = os.path.splitext(os.path.basename(path))[0]
        mol  = load_mol_from_sdf(path)
        if mol is None:
            print(f"  [WARN] Could not load {path}, skipping.")
            continue
        mol.SetProp("_Name", name)

        if os.path.basename(path) == template_name:
            template_mol = mol
            print(f"  [INFO] Template loaded: {path} "
                  f"({mol.GetNumConformers()} conformer(s))")
        else:
            molecules.append((name, mol))

    if template_mol is None:
        raise FileNotFoundError(
            f"Template '{template_name}' not found in '{folder}'"
        )
    print(f"  [INFO] {len(molecules)} probe molecule(s) loaded.")
    return template_mol, molecules


# ─────────────────────────────────────────────────────────────────────────────
# 3. ED ALIGNMENT  (shape-based O3A or MCS-constrained RMSD alignment)
# ─────────────────────────────────────────────────────────────────────────────

def align_mol_to_template(probe_mol, template_mol):
    """
    Align probe_mol to template_mol using crippen-O3A (open3D-like atom-based
    alignment available in RDKit without Open3D). 
    Falls back to MCS-constrained alignment if O3A gives a poor score.
    Returns (aligned_mol, rmsd).
    """
    probe    = Chem.RWMol(probe_mol)
    template = template_mol

    # Ensure both have 3-D coordinates
    if probe.GetNumConformers() == 0:
        AllChem.EmbedMolecule(probe, AllChem.ETKDGv3())
    if template.GetNumConformers() == 0:
        AllChem.EmbedMolecule(template, AllChem.ETKDGv3())

    # --- Crippen O3A alignment (no external dependency) ---
    pyO3A = AllChem.GetCrippenO3A(probe, template)
    pyO3A.Align()
    rmsd = rdMolAlign.CalcRMS(probe, template)

    # --- MCS fallback if RMSD is large (> 3 Å) ---
    if rmsd > 3.0:
        mcs_result = rdFMCS.FindMCS(
            [probe, template],
            bondCompare=rdFMCS.BondCompare.CompareAny,
            atomCompare=rdFMCS.AtomCompare.CompareAny,
            timeout=5
        )
        if mcs_result.numAtoms >= 4:
            mcs_mol   = Chem.MolFromSmarts(mcs_result.smartsString)
            probe_idx = probe.GetSubstructMatch(mcs_mol)
            tmpl_idx  = template.GetSubstructMatch(mcs_mol)
            if probe_idx and tmpl_idx:
                atom_map = list(zip(probe_idx, tmpl_idx))
                try:
                    rmsd = AllChem.AlignMol(probe, template, atomMap=atom_map)
                except Exception:
                    pass  # keep O3A result

    return probe.GetMol(), rmsd


# ─────────────────────────────────────────────────────────────────────────────
# 4. RMSD FILTRATION
# ─────────────────────────────────────────────────────────────────────────────

def rmsd_filter_conformers(mol, threshold: float = 1.0):
    """
    Given a molecule with multiple conformers, keep only those that are at
    least `threshold` Å apart from every already-kept conformer (greedy
    leader-picking). Returns a new molecule containing only the surviving
    conformers.
    """
    conf_ids = [c.GetId() for c in mol.GetConformers()]
    if not conf_ids:
        return mol

    kept   = [conf_ids[0]]
    pruned = 0

    for cid in conf_ids[1:]:
        accept = True
        for kid in kept:
            try:
                r = rdMolAlign.CalcRMS(mol, mol, cid, kid)
            except Exception:
                r = 0.0
            if r < threshold:
                accept = False
                break
        if accept:
            kept.append(cid)
        else:
            pruned += 1

    print(f"  [RMSD filter] kept {len(kept)}/{len(conf_ids)} conformers "
          f"(pruned {pruned} with RMSD < {threshold} Å)")

    # Rebuild mol with only surviving conformers
    filtered = Chem.RWMol(mol)
    all_ids  = {c.GetId() for c in mol.GetConformers()}
    for cid in all_ids - set(kept):
        filtered.RemoveConformer(cid)

    return filtered.GetMol()


# ─────────────────────────────────────────────────────────────────────────────
# 5. AGGREGATE PHARMACOPHORE FEATURES FROM ALL ALIGNED MOLECULES
# ─────────────────────────────────────────────────────────────────────────────

def aggregate_features(template_mol, aligned_mols, factory):
    """
    Collect pharmacophore features from the template and all aligned molecules.
    Returns a list of dicts: {family, type, pos, source_mol, conf_id}
    """
    all_features = []

    # Features from template
    tmpl_data = extract_pharmacophore_data(template_mol, factory)
    for cid, feats in tmpl_data.items():
        for f in feats:
            all_features.append({**f, 'source': 'template', 'conf_id': cid})

    # Features from aligned probes
    for name, mol in aligned_mols:
        ph_data = extract_pharmacophore_data(mol, factory)
        for cid, feats in ph_data.items():
            for f in feats:
                all_features.append({**f, 'source': name, 'conf_id': cid})

    return all_features


# ─────────────────────────────────────────────────────────────────────────────
# 6. CLUSTER FEATURES → PHARMACOPHORE POINTS
# ─────────────────────────────────────────────────────────────────────────────

def cluster_features(all_features, distance_cutoff: float = 1.5):
    """
    Group features of the same family that are within `distance_cutoff` Å
    of each other (single-linkage). Compute centroid and radius for each cluster.
    Returns list of dicts: {family, center, radius, count}
    """
    from collections import defaultdict

    by_family = defaultdict(list)
    for f in all_features:
        by_family[f['family']].append(np.array(f['pos']))

    pharmacophore_points = []

    for family, positions in by_family.items():
        positions = np.array(positions)
        assigned  = [-1] * len(positions)
        cluster_id = 0

        for i in range(len(positions)):
            if assigned[i] != -1:
                continue
            assigned[i] = cluster_id
            for j in range(i + 1, len(positions)):
                if assigned[j] == -1:
                    dist = np.linalg.norm(positions[i] - positions[j])
                    if dist <= distance_cutoff:
                        assigned[j] = cluster_id
            cluster_id += 1

        for cid in range(cluster_id):
            members = positions[[k for k, a in enumerate(assigned) if a == cid]]
            center  = members.mean(axis=0)
            radius  = float(np.max(np.linalg.norm(members - center, axis=1))) \
                      if len(members) > 1 else 0.5
            radius  = max(radius, 0.5)   # minimum sphere radius 0.5 Å

            pharmacophore_points.append({
                'family': family,
                'center': center.tolist(),
                'radius': round(radius, 3),
                'count':  len(members)
            })

    # Sort: most populated clusters first (more conserved = higher priority)
    pharmacophore_points.sort(key=lambda x: -x['count'])
    return pharmacophore_points


# ─────────────────────────────────────────────────────────────────────────────
# 7. SAVE PHARMACOPHORE MODEL AS XML
# ─────────────────────────────────────────────────────────────────────────────

def save_pharmacophore_xml(points, output_path: str, template_name: str):
    """
    Write the pharmacophore model to an XML file compatible with common
    pharmacophore tools (LigandScout-style schema).

    Schema:
    <pharmacophore name="..." template="...">
      <point id="1" family="Donor" x="..." y="..." z="..." radius="..." weight="..."/>
      ...
    </pharmacophore>
    """
    root = Element("pharmacophore")
    root.set("name",     os.path.splitext(output_path)[0])
    root.set("template", template_name)
    root.set("numPoints", str(len(points)))
    root.set("generated_by", "pharmacophore_pipeline.py")

    for idx, pt in enumerate(points, start=1):
        child = SubElement(root, "point")
        child.set("id",     str(idx))
        child.set("family", pt['family'])
        child.set("x",      f"{pt['center'][0]:.4f}")
        child.set("y",      f"{pt['center'][1]:.4f}")
        child.set("z",      f"{pt['center'][2]:.4f}")
        child.set("radius", f"{pt['radius']:.3f}")
        child.set("count",  str(pt['count']))
        # weight proportional to how many conformers/mols contributed
        child.set("weight", f"{min(1.0, pt['count'] / 10.0):.3f}")

    tree = ElementTree(root)
    try:
        indent(tree, space="  ")          # pretty-print (Python ≥ 3.9)
    except AttributeError:
        pass

    tree.write(output_path, encoding="unicode", xml_declaration=True)
    print(f"\n  [SAVED] Pharmacophore model → {output_path}")
    print(f"  [INFO]  {len(points)} pharmacophore point(s) written.")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(
    conformers_dir: str = CONFORMERS_DIR,
    template_name:  str = TEMPLATE_NAME,
    rmsd_threshold: float = RMSD_THRESHOLD,
    output_xml:     str = OUTPUT_XML
):
    print("=" * 60)
    print("  PHARMACOPHORE PIPELINE")
    print("=" * 60)

    # ── Build feature factory ─────────────────────────────────────
    print("\n[1] Building feature factory...")
    factory = ChemicalFeatures.BuildFeatureFactoryFromString(FEATURE_FACTORY_FDEF)
    print("    Feature families:", factory.GetFeatureFamilies())

    # ── Load molecules ────────────────────────────────────────────
    print(f"\n[2] Loading molecules from '{conformers_dir}'...")
    template_mol, probe_mols = load_all_conformers(conformers_dir, template_name)

    # ── RMSD-filter template conformers ──────────────────────────
    print(f"\n[3] RMSD filtration on template conformers (threshold={rmsd_threshold} Å)...")
    template_mol = rmsd_filter_conformers(template_mol, threshold=rmsd_threshold)

    # ── ED alignment + RMSD filter each probe ────────────────────
    print(f"\n[4] ED alignment + RMSD filtration of probe molecules...")
    aligned_mols = []
    for name, mol in probe_mols:
        print(f"  Aligning {name}...")
        aligned, rmsd = align_mol_to_template(mol, template_mol)
        print(f"    Alignment RMSD: {rmsd:.3f} Å")
        filtered = rmsd_filter_conformers(aligned, threshold=rmsd_threshold)
        aligned_mols.append((name, filtered))

    # ── Extract & aggregate pharmacophore features ────────────────
    print(f"\n[5] Extracting pharmacophore features from all molecules...")
    all_features = aggregate_features(template_mol, aligned_mols, factory)
    print(f"    Total raw features collected: {len(all_features)}")

    # ── Cluster features into pharmacophore points ────────────────
    print(f"\n[6] Clustering features into pharmacophore points...")
    ph_points = cluster_features(all_features, distance_cutoff=1.5)
    print(f"    Pharmacophore points identified: {len(ph_points)}")
    for pt in ph_points:
        x, y, z = pt['center']
        print(f"    {pt['family']:15s} center=({x:6.2f},{y:6.2f},{z:6.2f}) "
              f"r={pt['radius']:.2f} Å  count={pt['count']}")

    # ── Save XML ──────────────────────────────────────────────────
    print(f"\n[7] Saving pharmacophore model to XML...")
    save_pharmacophore_xml(ph_points, output_xml, template_name)

    print("\n" + "=" * 60)
    print("  PIPELINE COMPLETE")
    print("=" * 60)
    return ph_points


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(
        description="Pharmacophore ED-alignment + RMSD filtration pipeline"
    )
    parser.add_argument(
        "--conformers_dir", default=CONFORMERS_DIR,
        help=f"Folder with .sdf conformer files (default: {CONFORMERS_DIR})"
    )
    parser.add_argument(
        "--template", default=TEMPLATE_NAME,
        help=f"Template SDF filename inside conformers_dir (default: {TEMPLATE_NAME})"
    )
    parser.add_argument(
        "--rmsd", type=float, default=RMSD_THRESHOLD,
        help=f"RMSD threshold in Å for conformer pruning (default: {RMSD_THRESHOLD})"
    )
    parser.add_argument(
        "--output", default=OUTPUT_XML,
        help=f"Output XML filename (default: {OUTPUT_XML})"
    )
    args = parser.parse_args()

    run_pipeline(
        conformers_dir=args.conformers_dir,
        template_name=args.template,
        rmsd_threshold=args.rmsd,
        output_xml=args.output
    )